# Amazon Week 5 V2 — LambdaMART One-Click

Run **Runtime → Run all** in Google Colab.

When prompted, upload:
`amazon_week5_v1_feature_dataset.zip`


In [ ]:
# ============================================================
# AMAZON WEEK 5 V2 — LIGHTGBM LAMBDAMART ONE-CLICK
# Paste this ENTIRE block into ONE Google Colab cell and Run.
#
# INPUT:
#   amazon_week5_v1_feature_dataset.zip
#
# OUTPUT:
#   amazon_week5_v2_lambdamart_results.zip
#
# What this does:
#   1) loads the Week5 V1 feature table
#   2) preserves the fixed Week2 query-level train/test split
#   3) creates a query-level validation split from TRAIN only
#   4) tunes a small LambdaMART grid on validation NDCG@10
#   5) retrains the selected model on all 800 training queries
#   6) evaluates on the untouched 200-query test set
#   7) exports metrics, rankings, feature importance, model, plots
# ============================================================

!pip -q install "lightgbm>=4.0,<5.0" "scikit-learn>=1.4" pandas numpy matplotlib pyarrow

import os, json, shutil, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.metrics import ndcg_score
from google.colab import files

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

ROOT = Path("/content/week5_v2")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# STEP 1 — Upload Week5 V1 ZIP
# ------------------------------------------------------------
print("=" * 72)
print("STEP 1/8 — Upload amazon_week5_v1_feature_dataset.zip")
print("=" * 72)

uploaded = files.upload()
uploaded_names = list(uploaded.keys())

v1_zip = None
for name in uploaded_names:
    low = name.lower()
    if "week5" in low and "v1" in low and name.lower().endswith(".zip"):
        v1_zip = name
        break

if v1_zip is None:
    zip_candidates = [x for x in uploaded_names if x.lower().endswith(".zip")]
    if len(zip_candidates) == 1:
        v1_zip = zip_candidates[0]
    else:
        raise ValueError(
            "Could not identify the Week5 V1 ZIP. "
            "Please upload amazon_week5_v1_feature_dataset.zip."
        )

EXTRACT = ROOT / "v1"
EXTRACT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(v1_zip, "r") as z:
    z.extractall(EXTRACT)

print("✓ Loaded:", v1_zip)

def find_file(filename):
    hits = list(EXTRACT.rglob(filename))
    if not hits:
        hits = [
            p for p in EXTRACT.rglob("*")
            if p.is_file() and p.name.lower() == filename.lower()
        ]
    if not hits:
        return None
    return hits[0]

# ------------------------------------------------------------
# STEP 2 — Load + validate V1 feature table
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 2/8 — Load and validate Week5 V1 feature dataset")
print("=" * 72)

parquet_path = find_file("week5_feature_dataset.parquet")
csv_path = find_file("week5_feature_dataset.csv")
feature_list_path = find_file("week5_feature_list.json")

if parquet_path is not None:
    df = pd.read_parquet(parquet_path)
elif csv_path is not None:
    df = pd.read_csv(csv_path)
else:
    raise FileNotFoundError(
        "Could not find week5_feature_dataset.parquet or .csv in the ZIP."
    )

if feature_list_path is None:
    raise FileNotFoundError("Could not find week5_feature_list.json.")

with open(feature_list_path, "r", encoding="utf-8") as f:
    feature_cols = json.load(f)

required_cols = {
    "query_id", "product_id", "relevance_label", "split"
}
missing_required = required_cols - set(df.columns)
if missing_required:
    raise ValueError(
        "Missing required columns: " + ", ".join(sorted(missing_required))
    )

missing_features = [c for c in feature_cols if c not in df.columns]
if missing_features:
    raise ValueError(
        "Features listed in week5_feature_list.json are missing: "
        + ", ".join(missing_features)
    )

df["query_id"] = df["query_id"].astype(str)
df["product_id"] = df["product_id"].astype(str)
df["relevance_label"] = pd.to_numeric(
    df["relevance_label"], errors="raise"
).astype(int)

# Ensure model matrix is numeric.
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if df[feature_cols].isna().any().any():
    # LightGBM can handle NaN, but V1 should have been complete.
    print("Warning: NaN values detected; LightGBM will handle them natively.")

if df[["query_id", "product_id"]].duplicated().any():
    raise ValueError("Duplicate query-product pairs detected.")

train_all = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()

if len(train_all) == 0 or len(test) == 0:
    raise ValueError("Expected both train and test rows in the V1 dataset.")

print(f"Rows           : {len(df):,}")
print(f"Features       : {len(feature_cols)}")
print(f"Train queries  : {train_all['query_id'].nunique():,}")
print(f"Test queries   : {test['query_id'].nunique():,}")
print(f"Train rows     : {len(train_all):,}")
print(f"Test rows      : {len(test):,}")

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def sort_for_ranking(frame):
    return (
        frame
        .sort_values(["query_id", "product_id"], kind="mergesort")
        .reset_index(drop=True)
    )

def make_group(frame):
    # frame MUST already be sorted by query_id.
    return (
        frame.groupby("query_id", sort=False)
        .size()
        .astype(int)
        .tolist()
    )

def evaluate_ranking(frame, score_col, k=10):
    """
    Uses sklearn.ndcg_score so NDCG@10 is directly comparable
    to the Week2 evaluation implementation.
    """
    ndcgs = []
    recalls = []
    reciprocal_ranks = []

    for _, g in frame.groupby("query_id", sort=False):
        y_true = g["relevance_label"].to_numpy(dtype=float)
        y_score = g[score_col].to_numpy(dtype=float)

        # NDCG — sklearn tie handling matches the Week2 baseline evaluation.
        nd = ndcg_score(
            y_true.reshape(1, -1),
            y_score.reshape(1, -1),
            k=k
        )
        ndcgs.append(float(nd))

        order = np.argsort(-y_score, kind="mergesort")
        ranked_labels = y_true[order]
        topk = ranked_labels[:k]

        total_relevant = int((y_true > 0).sum())
        if total_relevant > 0:
            recalls.append(
                float((topk > 0).sum() / total_relevant)
            )
        else:
            recalls.append(0.0)

        rel_positions = np.where(ranked_labels > 0)[0]
        if len(rel_positions):
            reciprocal_ranks.append(
                float(1.0 / (rel_positions[0] + 1))
            )
        else:
            reciprocal_ranks.append(0.0)

    return {
        "NDCG@10": float(np.mean(ndcgs)),
        "Recall@10": float(np.mean(recalls)),
        "MRR": float(np.mean(reciprocal_ranks)),
        "Queries": int(frame["query_id"].nunique())
    }

def evaluate_ndcg_at(frame, score_col, k):
    vals = []
    for _, g in frame.groupby("query_id", sort=False):
        y = g["relevance_label"].to_numpy(dtype=float)
        s = g[score_col].to_numpy(dtype=float)
        vals.append(
            ndcg_score(
                y.reshape(1, -1),
                s.reshape(1, -1),
                k=k
            )
        )
    return float(np.mean(vals))

# ------------------------------------------------------------
# STEP 3 — Query-level TRAIN / VALIDATION split
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 3/8 — Create query-level validation split")
print("=" * 72)

train_query_ids = np.array(
    sorted(train_all["query_id"].unique())
)

rng = np.random.RandomState(SEED)
rng.shuffle(train_query_ids)

# 20% validation, entirely from the training side.
n_val = max(1, int(round(len(train_query_ids) * 0.20)))
val_ids = set(train_query_ids[:n_val])
fit_ids = set(train_query_ids[n_val:])

fit_df = train_all[
    train_all["query_id"].isin(fit_ids)
].copy()

val_df = train_all[
    train_all["query_id"].isin(val_ids)
].copy()

fit_df = sort_for_ranking(fit_df)
val_df = sort_for_ranking(val_df)
test = sort_for_ranking(test)

print(f"Fit queries       : {fit_df['query_id'].nunique()}")
print(f"Validation queries: {val_df['query_id'].nunique()}")
print(f"Test queries      : {test['query_id'].nunique()}")

# ------------------------------------------------------------
# STEP 4 — Small validation search
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 4/8 — Tune LambdaMART on validation NDCG@10")
print("=" * 72)

# This is intentionally small and reproducible rather than a huge grid.
search_space = []
for num_leaves in [7, 15, 31]:
    for min_child_samples in [5, 10, 20]:
        search_space.append({
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "learning_rate": 0.03,
            "n_estimators": 1200,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.00,
            "reg_lambda": 0.10,
        })

search_rows = []
best_model = None
best_val_ndcg = -np.inf
best_config = None
best_iteration = None

X_fit = fit_df[feature_cols]
y_fit = fit_df["relevance_label"]

X_val = val_df[feature_cols]
y_val = val_df["relevance_label"]

fit_group = make_group(fit_df)
val_group = make_group(val_df)

for i, cfg in enumerate(search_space, 1):
    print(
        f"[{i}/{len(search_space)}] "
        f"leaves={cfg['num_leaves']} "
        f"min_child={cfg['min_child_samples']}"
    )

    model = lgb.LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        random_state=SEED,
        n_jobs=-1,
        verbosity=-1,
        **cfg
    )

    model.fit(
        X_fit,
        y_fit,
        group=fit_group,
        eval_set=[(X_val, y_val)],
        eval_group=[val_group],
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            )
        ]
    )

    val_pred = model.predict(
        X_val,
        num_iteration=model.best_iteration_
    )

    temp_val = val_df.copy()
    temp_val["lambdamart_score"] = val_pred

    val_ndcg10 = evaluate_ndcg_at(
        temp_val,
        "lambdamart_score",
        10
    )

    val_ndcg3 = evaluate_ndcg_at(
        temp_val,
        "lambdamart_score",
        3
    )

    row = {
        "num_leaves": cfg["num_leaves"],
        "min_child_samples": cfg["min_child_samples"],
        "learning_rate": cfg["learning_rate"],
        "best_iteration": int(model.best_iteration_),
        "validation_NDCG@10": val_ndcg10,
        "validation_NDCG@3": val_ndcg3
    }
    search_rows.append(row)

    print(
        f"    best_iter={model.best_iteration_:>4} | "
        f"val NDCG@10={val_ndcg10:.6f}"
    )

    if val_ndcg10 > best_val_ndcg:
        best_val_ndcg = val_ndcg10
        best_model = model
        best_config = cfg.copy()
        best_iteration = int(model.best_iteration_)

search_df = (
    pd.DataFrame(search_rows)
    .sort_values(
        ["validation_NDCG@10", "validation_NDCG@3"],
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nBest validation configuration:")
print(search_df.head(1).to_string(index=False))

# ------------------------------------------------------------
# STEP 5 — Retrain selected LambdaMART on ALL train queries
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 5/8 — Retrain final LambdaMART on all training queries")
print("=" * 72)

train_final = sort_for_ranking(train_all)

final_params = best_config.copy()
final_params["n_estimators"] = max(1, best_iteration)

final_model = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
    **final_params
)

final_model.fit(
    train_final[feature_cols],
    train_final["relevance_label"],
    group=make_group(train_final)
)

test["lambdamart_score"] = final_model.predict(
    test[feature_cols]
)

# ------------------------------------------------------------
# STEP 6 — Evaluate test set + baseline score columns
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 6/8 — Evaluate untouched test set")
print("=" * 72)

metric_rows = []

# LambdaMART
lm_metrics = evaluate_ranking(
    test,
    "lambdamart_score",
    k=10
)
lm_metrics["model"] = "LambdaMART"
lm_metrics["NDCG@3"] = evaluate_ndcg_at(
    test, "lambdamart_score", 3
)
metric_rows.append(lm_metrics)

# Existing score-based baselines available in the V1 table.
baseline_cols = [
    ("BM25", "bm25_score"),
    ("TF-IDF", "tfidf_similarity"),
    ("SBERT", "sbert_similarity"),
    ("CLIP", "clip_similarity")
]

for model_name, col in baseline_cols:
    if col in test.columns:
        m = evaluate_ranking(test, col, k=10)
        m["model"] = model_name
        m["NDCG@3"] = evaluate_ndcg_at(
            test, col, 3
        )
        metric_rows.append(m)

metrics_df = pd.DataFrame(metric_rows)[
    [
        "model",
        "NDCG@10",
        "NDCG@3",
        "Recall@10",
        "MRR",
        "Queries"
    ]
]

bm25_ndcg = None
if "BM25" in set(metrics_df["model"]):
    bm25_ndcg = float(
        metrics_df.loc[
            metrics_df["model"] == "BM25",
            "NDCG@10"
        ].iloc[0]
    )
    metrics_df["NDCG_vs_BM25_%"] = (
        (metrics_df["NDCG@10"] - bm25_ndcg)
        / bm25_ndcg
        * 100
    )
else:
    metrics_df["NDCG_vs_BM25_%"] = np.nan

metrics_df = metrics_df.sort_values(
    "NDCG@10",
    ascending=False
).reset_index(drop=True)

print("\nTEST METRICS")
print(metrics_df.to_string(index=False))

# Reference value from the Week2 artifact that created this V1 dataset.
# It is written as a reference, not used for training/tuning.
WEEK2_XGB_NDCG10_REFERENCE = 0.922169

lm_ndcg = float(
    metrics_df.loc[
        metrics_df["model"] == "LambdaMART",
        "NDCG@10"
    ].iloc[0]
)

vs_week2_xgb_pct = (
    (lm_ndcg - WEEK2_XGB_NDCG10_REFERENCE)
    / WEEK2_XGB_NDCG10_REFERENCE
    * 100
)

print(
    f"\nLambdaMART vs Week2 XGBoost reference "
    f"({WEEK2_XGB_NDCG10_REFERENCE:.6f}): "
    f"{vs_week2_xgb_pct:+.3f}%"
)

# Note:
# Recall@10 will often be 1.0 because each judged candidate set has only
# about 5–8 products. NDCG@10 and MRR remain meaningful ordering metrics.

# ------------------------------------------------------------
# STEP 7 — Feature importance + ranking outputs + plots
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 7/8 — Export model diagnostics")
print("=" * 72)

OUT = ROOT / "output"
OUT.mkdir(parents=True, exist_ok=True)

# Feature importance
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_gain": final_model.booster_.feature_importance(
        importance_type="gain"
    ),
    "importance_split": final_model.booster_.feature_importance(
        importance_type="split"
    )
})

gain_sum = importance_df["importance_gain"].sum()
if gain_sum > 0:
    importance_df["gain_pct"] = (
        importance_df["importance_gain"]
        / gain_sum
        * 100
    )
else:
    importance_df["gain_pct"] = 0.0

importance_df = importance_df.sort_values(
    "importance_gain",
    ascending=False
).reset_index(drop=True)

# Test ranking order
test_rankings = test.copy()
test_rankings["predicted_rank"] = (
    test_rankings
    .groupby("query_id")["lambdamart_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

test_rankings = test_rankings.sort_values(
    ["query_id", "predicted_rank"]
)

keep_cols = [
    "query_id",
    "query",
    "product_id",
    "title",
    "relevance_label",
    "lambdamart_score",
    "predicted_rank",
    "bm25_score",
    "tfidf_similarity",
    "sbert_similarity",
    "clip_similarity"
]
keep_cols = [c for c in keep_cols if c in test_rankings.columns]

# Save model
final_model.booster_.save_model(
    str(OUT / "week5_v2_lambdamart_model.txt")
)

metrics_df.to_csv(
    OUT / "week5_v2_test_metrics.csv",
    index=False
)

search_df.to_csv(
    OUT / "week5_v2_validation_search.csv",
    index=False
)

importance_df.to_csv(
    OUT / "week5_v2_feature_importance.csv",
    index=False
)

test_rankings[keep_cols].to_csv(
    OUT / "week5_v2_test_rankings.csv",
    index=False
)

test_rankings.to_parquet(
    OUT / "week5_v2_test_rankings.parquet",
    index=False
)

pd.DataFrame({
    "query_id": sorted(fit_ids)
}).to_csv(
    OUT / "week5_v2_fit_query_ids.csv",
    index=False
)

pd.DataFrame({
    "query_id": sorted(val_ids)
}).to_csv(
    OUT / "week5_v2_validation_query_ids.csv",
    index=False
)

config_output = {
    "seed": SEED,
    "objective": "lambdarank",
    "primary_metric": "NDCG@10",
    "num_features": len(feature_cols),
    "feature_cols": feature_cols,
    "fit_queries_during_tuning": len(fit_ids),
    "validation_queries": len(val_ids),
    "final_train_queries": int(train_final["query_id"].nunique()),
    "test_queries": int(test["query_id"].nunique()),
    "best_validation_ndcg10": float(best_val_ndcg),
    "best_iteration": int(best_iteration),
    "selected_hyperparameters": best_config,
    "final_hyperparameters": final_params,
    "week2_xgboost_ndcg10_reference": WEEK2_XGB_NDCG10_REFERENCE,
    "lambdamart_test_ndcg10": lm_ndcg,
    "lambdamart_vs_week2_xgb_pct": float(vs_week2_xgb_pct)
}

with open(
    OUT / "week5_v2_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(config_output, f, indent=2)

# Plot 1 — Feature importance
top_imp = importance_df.head(15).sort_values(
    "importance_gain",
    ascending=True
)

plt.figure(figsize=(9, 6))
plt.barh(
    top_imp["feature"],
    top_imp["importance_gain"]
)
plt.xlabel("LightGBM Gain Importance")
plt.ylabel("Feature")
plt.title("Week 5 V2 — LambdaMART Feature Importance")
plt.tight_layout()
plt.savefig(
    OUT / "week5_v2_feature_importance.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

# Plot 2 — NDCG@10 comparison
plot_metrics = metrics_df.sort_values(
    "NDCG@10",
    ascending=True
)

plt.figure(figsize=(8, 5))
plt.barh(
    plot_metrics["model"],
    plot_metrics["NDCG@10"]
)
plt.xlabel("NDCG@10")
plt.ylabel("Model / Signal")
plt.title("Week 5 V2 — Test NDCG@10")
plt.xlim(
    max(0, plot_metrics["NDCG@10"].min() - 0.05),
    min(1.0, plot_metrics["NDCG@10"].max() + 0.03)
)
plt.tight_layout()
plt.savefig(
    OUT / "week5_v2_ndcg_comparison.png",
    dpi=180,
    bbox_inches="tight"
)
plt.close()

# README
best_row = search_df.iloc[0]

readme = f"""
AMAZON SEARCH / LTR — WEEK 5 V2 LAMBDAMART
==========================================

DATA
----
Rows: {len(df):,}
Total queries: {df['query_id'].nunique():,}
Fixed training queries: {train_all['query_id'].nunique():,}
Fixed test queries: {test['query_id'].nunique():,}
Features: {len(feature_cols)}

MODEL
-----
LightGBM LGBMRanker
Objective: LambdaRank / LambdaMART
Primary evaluation: NDCG@10

VALIDATION
----------
The original Week2 test split was not used for tuning.

From the 800 fixed training queries:
- Fit during tuning: {len(fit_ids)}
- Validation: {len(val_ids)}

Selected:
- num_leaves: {int(best_row['num_leaves'])}
- min_child_samples: {int(best_row['min_child_samples'])}
- best_iteration: {int(best_row['best_iteration'])}
- validation NDCG@10: {float(best_row['validation_NDCG@10']):.6f}

FINAL TEST
----------
LambdaMART NDCG@10: {lm_ndcg:.6f}
LambdaMART vs Week2 XGBoost reference ({WEEK2_XGB_NDCG10_REFERENCE:.6f}):
{vs_week2_xgb_pct:+.3f}%

IMPORTANT ROADMAP NOTE
----------------------
The original Week5 roadmap says:
"LambdaMART >=12% above Week2 pointwise baseline."

Week2 XGBoost NDCG@10 = 0.922169.
A 12% relative increase would require NDCG@10 > 1.0,
which is mathematically impossible.

Therefore the defensible Week5 criterion is:
1. LambdaMART should be competitive with / outperform the Week2 pointwise model;
2. LambdaMART should materially outperform BM25;
3. feature importance and later ablations should explain which feature groups help.

METRIC NOTE
-----------
Recall@10 can equal 1.0 for nearly every method because the judged candidate
lists contain only about 5–8 products per query, fewer than the cutoff of 10.
For ranking quality, NDCG@10 and MRR are therefore more informative here.

NEXT
----
Week5 V3:
- formal model comparison
- Week2 XGBoost vs LambdaMART
- ablation of lexical / SBERT / CLIP / metadata groups
""".strip()

with open(
    OUT / "README_WEEK5_V2.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(readme + "\n")

# ------------------------------------------------------------
# STEP 8 — ZIP + auto-download
# ------------------------------------------------------------
print("\n" + "=" * 72)
print("STEP 8/8 — Package Week5 V2 results")
print("=" * 72)

zip_out = Path(
    "/content/amazon_week5_v2_lambdamart_results.zip"
)

if zip_out.exists():
    zip_out.unlink()

with zipfile.ZipFile(
    zip_out,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in OUT.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.name)

print("\n✅ WEEK 5 V2 COMPLETE")
print("✅ Created:", zip_out)
print("\nUpload this ZIP back to ChatGPT for Week5 V3.")
files.download(str(zip_out))
